# Economic Development, Life Expectancy, and CO₂ Emissions📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
- I would like to analyze how the per capita GDP, the life expectancy,
and the CO₂ emission per person have changed in Nepal, India, the United
States, the United Kingdom, and China between 2010 and 2023. The two
countries that I want to look into are Nepal and India due to my personal
connection with them.

The problem lies in how the economic and health development can affect the environment. These five countries represent quite different economies, and analyzing
how similar measures change for all of them can be interesting. This
research will show some patterns and relationships; no cause-and-effect
claims will be made about the GDP and changes in other measures.

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
- My main research question is: How have the trends in GDP per capita,
life expectancy, and CO₂ emissions per capita developed together in
these five countries between 2010 and 2023?

I am going to address these particular questions:

1. What were the trends in each variable in each country?
2. What countries in the most recent year for which I have data have had
   life expectancies that are high or low compared to their GDP per
   capita?
3. Within each country, what years that have had high GDP per capita also
   have had high CO₂ emissions per capita?

## What would an answer look like?
*What is your hypothesized answer to your question?*
- GDP per capita and life expectancy will tend to correlate positively; however,
there might be differences in how GDP and CO₂ emissions correlate. This is a
hypothesis, not a conclusion.

The response will consist of three line graphs with years on the horizontal
axis and one variable on the vertical axis. Each country will have its separate
line. In addition, there will be a scatterplot for GDP per capita vs life
expectancy, with GDP on the horizontal axis and life expectancy on the
vertical axis and different colors representing different countries. Another
scatterplot will be made for GDP per capita and CO₂ emissions per person.

Specific details revealed by the graphs will be pointed out; complete data
for some years will be indicated, along with missing data points for some
countries.

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*

| Dataset | Type | Fields I will use | Source |
| --- | --- | --- | --- |
| GDP per capita, constant 2015 US$ | World Bank JSON API | Country code, year, GDP per capita | https://data.worldbank.org/indicator/NY.GDP.PCAP.KD |
| Life expectancy | Our World in Data CSV file | Code, Year, life expectancy | https://ourworldindata.org/grapher/life-expectancy |
| CO₂ emissions per capita | Our World in Data CSV file | Code, Year, tonnes per person | https://ourworldindata.org/grapher/co-emissions-per-capita |

These are three distinct datasets and two data types: JSON from an API and
CSV files. I will join them on three-letter country code and calendar year.
For example, Nepal's 2020 GDP record will match Nepal's 2020 life expectancy
and CO₂ records. I will check for missing values and retain the country-years
where all three measures are available.

## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*

- I'll get the GDP per capita from the World Bank API, and I'll import
life expectancy and per capita carbon dioxide emissions from two CSV
datasets. I'll filter the data by Nepal, India, the United States,
the United Kingdom, and China for the period 2010–2023. And then I'll
merge the datasets by the country codes and years and check for missing
values among the merged rows.

Here is the code that demonstrates the joining of all three sources
in one dataset. It shows that they can be joined, which will be used
for creating the visualizations at a further stage of the work.

In [3]:
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import requests

COUNTRIES = ["NPL", "IND", "USA", "GBR", "CHN"]
START_YEAR = 2010
END_YEAR = 2023

# Source 1: World Bank JSON API
world_bank_url = (
    "https://api.worldbank.org/v2/country/all/"
    "indicator/NY.GDP.PCAP.KD"
)

response = requests.get(
    world_bank_url,
    params={
        "format": "json",
        "date": f"{START_YEAR}:{END_YEAR}",
        "per_page": 10000,
    },
    timeout=60,
)
response.raise_for_status()
world_bank_result = response.json()

gdp_df = pd.DataFrame(
    {
        "Code": item["countryiso3code"],
        "Year": int(item["date"]),
        "gdp_per_capita": item["value"],
    }
    for item in world_bank_result[1]
    if item["countryiso3code"] in COUNTRIES
)

print("World Bank GDP rows:", len(gdp_df))
display(gdp_df.head())

World Bank GDP rows: 70


,Code,Year,gdp_per_capita
0,CHN,2023,12483.979110
1,CHN,2022,11830.298768
2,CHN,2021,11469.317451
3,CHN,2020,10573.379406
4,CHN,2019,10356.292489


In [5]:
# Source 2: Life expectancy CSV
life_url = "https://ourworldindata.org/grapher/life-expectancy.csv"

life_raw = pd.read_csv(
    life_url,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

life_df = life_raw[["Code", "Year", life_raw.columns[3]]].copy()
life_df.columns = ["Code", "Year", "life_expectancy"]

life_df = life_df[
    life_df["Code"].isin(COUNTRIES)
    & life_df["Year"].between(START_YEAR, END_YEAR)
]

print("Life expectancy rows:", len(life_df))
display(life_df.head())

Life expectancy rows: 70


,Code,Year,life_expectancy
3655,CHN,2010,75.6679
3656,CHN,2011,75.8891
3657,CHN,2012,76.2000
3658,CHN,2013,76.4521
3659,CHN,2014,76.7143


In [6]:
# Source 3: CO2 emissions per capita CSV
co2_url = "https://ourworldindata.org/grapher/co-emissions-per-capita.csv"

co2_raw = pd.read_csv(
    co2_url,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

co2_df = co2_raw[["Code", "Year", co2_raw.columns[3]]].copy()
co2_df.columns = ["Code", "Year", "co2_tonnes_per_person"]

co2_df = co2_df[
    co2_df["Code"].isin(COUNTRIES)
    & co2_df["Year"].between(START_YEAR, END_YEAR)
]

print("CO2 rows:", len(co2_df))
display(co2_df.head())

CO2 rows: 70


,Code,Year,co2_tonnes_per_person
4999,CHN,2010,6.370445
5000,CHN,2011,6.998823
5001,CHN,2012,7.131918
5002,CHN,2013,7.209799
5003,CHN,2014,7.187588


In [7]:
project_df = (
    gdp_df
    .merge(life_df, on=["Code", "Year"], how="inner")
    .merge(co2_df, on=["Code", "Year"], how="inner")
    .dropna()
    .sort_values(["Code", "Year"])
)

print("Complete country-year rows:", len(project_df))
print("Countries:", sorted(project_df["Code"].unique()))
print("Years:", project_df["Year"].min(), "to", project_df["Year"].max())
display(project_df.head(10))

Complete country-year rows: 70
Countries: ['CHN', 'GBR', 'IND', 'NPL', 'USA']
Years: 2010 to 2023


,Code,Year,gdp_per_capita,life_expectancy,co2_tonnes_per_person
13,CHN,2010,5764.650124,75.6679,6.370445
12,CHN,2011,6275.716343,75.8891,6.998823
11,CHN,2012,6723.090037,76.2000,7.131918
10,CHN,2013,7197.937319,76.4521,7.209799
9,CHN,2014,7686.373695,76.7143,7.187588
8,CHN,2015,8175.332851,76.9755,7.060955
7,CHN,2016,8679.225288,77.2140,6.942884
6,CHN,2017,9221.350456,77.2353,7.080384
5,CHN,2018,9798.578906,77.7090,7.291561
4,CHN,2019,10356.292489,77.9396,7.526071


## Resources and References
*What resources and references have you used for this project?*
- World Bank, GDP per capita (constant 2015 US$):
  https://data.worldbank.org/indicator/NY.GDP.PCAP.KD
- Our World in Data, Life expectancy:
  https://ourworldindata.org/grapher/life-expectancy
- Our World in Data, CO₂ emissions per capita:
  https://ourworldindata.org/grapher/co-emissions-per-capita
- World Bank Indicators API documentation:
  https://datahelpdesk.worldbank.org/knowledgebase/articles/898581

In [2]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

[NbConvertApp] Converting notebook source.ipynb to python
[NbConvertApp] Writing 1261 bytes to source.py
